In [24]:
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(df.shape)
df.columns.tolist()

(1470, 35)


['Age',
 'Attrition',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EmployeeCount',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'Over18',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StandardHours',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [25]:
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"],
             errors="ignore")
df["AttritionFlag"] = (df["Attrition"] == "Yes").astype(int)

def attr_rate(col):
    return df.groupby(col, observed=True)["AttritionFlag"].mean().mul(100).round(1)

df.shape

(1470, 32)

In [26]:
overall = df["AttritionFlag"].mean() * 100
print("Employees:", len(df))
print("Left the company:", df["AttritionFlag"].sum())
print("Overall attrition rate: %.1f%%" % overall)

print()
ot = attr_rate("OverTime")
print(ot)
print("Ratio (overtime / no overtime): %.2f" % (ot["Yes"] / ot["No"]))

print()
print(attr_rate("Department").sort_values(ascending=False))
print()
print(attr_rate("JobRole").sort_values(ascending=False))

Employees: 1470
Left the company: 237
Overall attrition rate: 16.1%

OverTime
No     10.4
Yes    30.5
Name: AttritionFlag, dtype: float64
Ratio (overtime / no overtime): 2.93

Department
Sales                     20.6
Human Resources           19.0
Research & Development    13.8
Name: AttritionFlag, dtype: float64

JobRole
Sales Representative         39.8
Laboratory Technician        23.9
Human Resources              23.1
Sales Executive              17.5
Research Scientist           16.1
Healthcare Representative     6.9
Manufacturing Director        6.9
Manager                       4.9
Research Director             2.5
Name: AttritionFlag, dtype: float64


**Findings**
1. **Overall:** 237 of 1,470 employees left, an overall attrition rate of 16.1%.
2. **Overtime:** Employees working overtime left at 30.5%, against 10.4% for those who did not, about 3x higher (ratio 2.93).
3. **Department and role:** Sales has the highest department attrition (20.6%), followed by HR (19.0%) and R&D (13.8%). Sales Representatives are the highest-risk role at 39.8%, and Research Directors are the lowest at 2.5%. Some role groups are small, so treat these as patterns, not exact figures.

In [27]:
df["AgeGroup"] = pd.cut(df["Age"], bins=[17, 25, 35, 45, 55, 70],
                        labels=["18-25", "26-35", "36-45", "46-55", "56+"])
df["IncomeBand"] = pd.qcut(df["MonthlyIncome"], 4,
                           labels=["Low", "Lower-mid", "Upper-mid", "High"])
sat_map = {1: "Low", 2: "Medium", 3: "High", 4: "Very High"}
df["JobSatisfactionLabel"] = df["JobSatisfaction"].map(sat_map)

print("AGE GROUP");     print(attr_rate("AgeGroup")); print()
print("INCOME BAND");   print(attr_rate("IncomeBand")); print()
order = ["Low", "Medium", "High", "Very High"]
print("JOB SATISFACTION"); print(attr_rate("JobSatisfactionLabel").reindex(order)); print()
corr = df.select_dtypes("number").corr()["AttritionFlag"].drop("AttritionFlag")
print("CORRELATION WITH ATTRITION"); print(corr.sort_values().round(2))

AGE GROUP
AgeGroup
18-25    35.8
26-35    19.1
36-45     9.2
46-55    11.5
56+      17.0
Name: AttritionFlag, dtype: float64

INCOME BAND
IncomeBand
Low          29.3
Lower-mid    14.2
Upper-mid    10.6
High         10.3
Name: AttritionFlag, dtype: float64

JOB SATISFACTION
JobSatisfactionLabel
Low          22.8
Medium       16.4
High         16.5
Very High    11.3
Name: AttritionFlag, dtype: float64

CORRELATION WITH ATTRITION
TotalWorkingYears          -0.17
JobLevel                   -0.17
YearsInCurrentRole         -0.16
MonthlyIncome              -0.16
Age                        -0.16
YearsWithCurrManager       -0.16
StockOptionLevel           -0.14
YearsAtCompany             -0.13
JobInvolvement             -0.13
JobSatisfaction            -0.10
EnvironmentSatisfaction    -0.10
WorkLifeBalance            -0.06
TrainingTimesLastYear      -0.06
DailyRate                  -0.06
RelationshipSatisfaction   -0.05
YearsSinceLastPromotion    -0.03
Education                  -0.03
Percent

**Findings**
1. **Age:** Employees aged 18-25 leave most (35.8%), about 4x the 36-45 group (9.2%).
2. **Pay:** The lowest income band leaves at 29.3%, against 10.3% for the highest band, about 3x.
3. **Job satisfaction:** Employees with low satisfaction left at 22.8%, against 11.3% for very high satisfaction, about 2x.
4. **Correlations:** All numeric correlations with attrition are weak (strongest: -0.17 for total working years and job level). No single numeric factor explains attrition; overtime, age and pay band are clearer signals. Correlation is not causation.


In [28]:
df = df.drop(columns=["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"],
             errors="ignore")
df["AttritionFlag"] = (df["Attrition"] == "Yes").astype(int)

def attr_rate(col):
    return df.groupby(col, observed=True)["AttritionFlag"].mean().mul(100).round(1)

## Summary and recommendations
**Key findings**
1. Overall attrition is 16.1% (237 of 1,470 employees).
2. Overtime employees leave at 30.5% vs 10.4% for others, about 3x.
3. Highest-risk groups: Sales Representatives (39.8%), ages 18-25 (35.8%) and the lowest income band (29.3%).
4. Low job satisfaction roughly doubles attrition (22.8% vs 11.3%).

**Recommendations**
1. Manage overtime, starting with Sales Representatives and Laboratory Technicians.
2. Retain early-career staff with mentoring, a clear promotion path and a pay review for entry-level roles.
3. Review workload and incentives for Sales Representatives and run regular satisfaction check-ins.

**Limitations:** This is a single static dataset, so these are patterns, not proof of cause. Age, pay and job level overlap.
